In [2]:
import asyncio

async def say_hello(name, delay):
    print(f"[{name}] Starting...")
    await asyncio.sleep(delay)
    print(f"[{name}] Finished!")
    return f"Hello {name}"

# 顺序执行
async def sequential():
    result1 = await say_hello("task1", 1)
    result2 = await say_hello("task2", 1)
    return result1, result2

# 并发执行
async def concurrent():
    # asyncio.gather: 并发执行多个协程
    reuslts = await asyncio.gather(
        say_hello("taskA", 2),
        say_hello("taskB", 1),
        say_hello("taskC", 1.5)
    )
    return reuslts

async def main():
    print("=== 顺序执行 ===")
    import time 
    start_time = time.time()
    await sequential()
    print(f"耗时：{time.time() - start_time:.2f}秒\n")

    print("=== 并发执行 ===")
    start_time = time.time()
    await concurrent()
    print(f"耗时：{time.time() - start_time:.2f}秒")


await main()


=== 顺序执行 ===
[task1] Starting...
[task1] Finished!
[task2] Starting...
[task2] Finished!
耗时：2.00秒

=== 并发执行 ===
[taskA] Starting...
[taskB] Starting...
[taskC] Starting...
[taskB] Finished!
[taskC] Finished!
[taskA] Finished!
耗时：2.00秒


In [10]:
# semaphore
async def demo_semaphore():
    semaphore = asyncio.Semaphore(2)

    async def task(name):
        async with semaphore:
            print(f"{name} Start")
            await asyncio.sleep(1)
            print(f"{name} Finished")
    
    await asyncio.gather(
        task("task1"),
        task("task2"),
        task("task3"),
        task("task4")
    )

In [11]:
await demo_semaphore()

task1 Start
task2 Start
task1 Finished
task2 Finished
task3 Start
task4 Start
task3 Finished
task4 Finished


In [12]:
from functools import lru_cache
from typing import Optional

# 单例模式的配置类
class Settings:
    def __init__(self) -> None:
        self.dashscope_api_key = "your_api_key"
        self.default_model = "qwen-plus-1.5"
        self.default_temperature = 0.7

# 使用lru_cache实现单例模式
@lru_cache(maxsize=1)
def get_settings() ->Settings:
    return Settings()


In [13]:
# LLM客户端类
class LLMClient:
    def __init__(
        self,
        api_key: str,
        model: str
    ) -> None:
        self.api_key = api_key
        self.model = model
        print(f"创建LLMClient实例：{model}")


In [14]:
# 依赖注入函数
def get_llm_client() -> LLMClient:
    settings = get_settings()
    return LLMClient(
        api_key=settings.dashscope_api_key,
        model=settings.default_model
    )


In [15]:
# 测试单例模式
client1 = get_llm_client()
client2 = get_llm_client()
print(f"是同一个实例：{client1 is client2}")

创建LLMClient实例：qwen-plus-1.5
创建LLMClient实例：qwen-plus-1.5
是同一个实例：False


In [ ]:
# FastAPI依赖注入示例
from fastapi import FastAPI, Depends, HTTPException
from typing import Annotated

from gevent import getcurrent


# 数据库连接类
class Database:
    def __init__(self) -> None:
        self.connected = True

# 依赖项：获取数据库连接
def get_db() -> Database:
    db = Database()
    try:
        yield db
    finally:
        pass

# 依赖项：获取当前用户
def get_current_user(token: str = Depends(lambda: "fake-token")):
    if token != "valid-token":
        raise HTTPException(status_code=401, detail="无效的token")
    return {"user_id": 123, "name": "Alice"}

# create FastAPI app
app = FastAPI()

# 使用依赖注入
@app.get("/profile")
async def get_profile(
    current_user: dict = Depends(get_current_user),     # 注入用户
    db: Database = Depends(get_db)  # 注入数据库
):
    return {
        "user": current_user,
        "db_connected": db.connected
    }

In [ ]:
from contextlib import asynccontextmanager
import logging

from fastapi import FastAPI

from learn_20260210.app.config import settings
from learn_20260210.app.config.constants import API_PREFIX
from learn_20260210.app.utils.logger import LoggerConfig
from learn_20260210.app.api.routers import knowledge_base_router, script_recommender_router, quality_inspector_router


# 配置日志
LoggerConfig.setup_logger()
logger = logging.getLogger(__name__)


@asynccontextmanager
async def lifespan(app: FastAPI):
    """应用生命周期管理"""
    # 启动时执行
    logger.info("=" * 60)
    logger.info("大模型客服助手服务启动中...")
    logger.info(f"应用名称: {settings.APP_NAME}")
    logger.info(f"应用版本: {settings.APP_VERSION}")
    logger.info(f"运行环境: {settings.APP_ENV}")
    logger.info("=" * 60)
    
    yield
    
    # 关闭时执行
    logger.info("大模型客服助手服务关闭")

app = FastAPI(
    title=settings.APP_NAME,
    version=settings.APP_VERSION,
    description="大模型客服助手",
    lifespan=lifespan, # 生命周期管理
)

# 注册路由
app.include_router(knowledge_base_router, prefix=API_PREFIX)
app.include_router(script_recommender_router, prefix=API_PREFIX)
app.include_router(quality_inspector_router, prefix=API_PREFIX)

In [ ]:
from fastapi import Path, Query, Body, Header, params
from pydantic import BaseModel
from tomlkit import item

# path parameter
@app.get("/search")
async def get_item(item_id: int = Path(..., ge=1)):
    # item_id: int 自动类型转换
    # Path(...): 必填参数，ge=1: 参数必须大于等于1
    return {"item_id": item_id}

# search?query=xxx
@app.get("/search")
async def search(q: str = Query(..., min_length=1), page: int = Query(1, ge=1)):
    # q: str 自动类型转换，Query(...): 必填参数，min_length=1: 最小长度为1
    # page: int 自动类型转换，Query(1): 默认值为1，ge=1: 参数必须大于等于1
    return {"query": q, "page": page}

# request body  
class Item(BaseModel):
    
    name: str
    price: float

@app.post("/items")
async def create_item(item: Item):
    return item

# Header parameter
@app.get("/headers")
async def get_headers(token: str = Header(...)):
    return {"token": token}



In [2]:
# 响应模型

from fastapi import Response, FastAPI
from typing import Optional
from pydantic import BaseModel

app = FastAPI()

class ItemResponse(BaseModel):
    id: int
    name: str
    price: float
    description: Optional[str] = None

@app.get("/items/{item_id}", response_model=ItemResponse)
async def get_item(item_id: int):
    return {
        "id": item_id,
        "name": "test",
        "price": 9.99,
    }



In [ ]:
# 中间件
from fastapi import FastAPI, Request
import time

app = FastAPI()

# 记录请求耗时的中间件
@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start_time = time.time()
    response = await call_next(request)
    process_time = time.time() - start_time
    response.headers["X-Process-Time"] = str(process_time)
    return response

In [3]:
app.include_router(
    knowledge_base_router,
    prefix=API_PREFIX,
    tags=["知识库"]
)

from fastapi import APIRouter
from websockets import route

router = APIRouter()

@router.get("/knowledge/search")
async def search_knowledge(query: str):
    return {"query": query}


NameError: name 'knowledge_base_router' is not defined

In [ ]:
from fastapi import HTTPException

@app.get("/items/{item_id}")
async def get_item(item_id= int):
    if item_id not in items:
        raise HTTPException(
            status_code=404,
            detail=f"Item {item_id} not found",
            headers={"X-Error": "NotFound"}
        )
    return items[item_id]

# 全局异常处理
@app.excetion_handler(Exception)
async def global_exception_handler(request, exc):
    return {
        "message": "Internal Server Error"
    }

In [ ]:
from contextlib import asynccontextmanager
from os import close
from time import clock_getres
from fastapi import FastAPI

@asynccontextmanager
async def lifespan(app: FastAPI):
    # excute when start
    print("service is starting...")
    await init_db()
    await init_cache()
    yield # state of handling request
    
    # excute when shutdown
    print("service is shutting down...")
    await close_db()

app = FastAPI(lifespan=lifespan)